In [9]:
import random
import pickle
from collections import defaultdict

In [ ]:
class MemoryPairsGame:
    def __init__(self, n_pairs=8, seed=None):
        self.n_pairs = n_pairs
        self.rng = random.Random(seed)
        self.reset()

    def reset(self):
        self.deck = list(range(self.n_pairs)) * 2
        self.rng.shuffle(self.deck)
        self.matched = set()
        return self

    def hidden_positions(self):
        return [i for i in range(len(self.deck)) if i not in self.matched]

    def reveal(self, position):
        return self.deck[position]

    def match(self, pos1, pos2):
        if self.deck[pos1] == self.deck[pos2]:
            self.matched.add(pos1)
            self.matched.add(pos2)
            return True
        return False

    def done(self):
        return len(self.matched) == len(self.deck)

    def draw(self, temporary_reveals=None):
        temporary_reveals = temporary_reveals or {}
        cells = []
        for position, value in enumerate(self.deck):
            if position in self.matched:
                cells.append(f'[{value}]')
            elif position in temporary_reveals:
                cells.append(f' {temporary_reveals[position]} ')
            else:
                cells.append(f' {position:02d} ')
        print(' | '.join(cells))

In [10]:
class MemoryPairAgent:
    def __init__(self, c=1.4, seed=None):
        self.c = c
        self.rng = random.Random(seed)
        self.reset()

    def reset(self):
        self.seen = {}
        self.by_value = defaultdict(set)
        self.matched = set()
        self.action_counts = defaultdict(int)
        self.action_values = defaultdict(float)
        self.total_actions = 0

    def observe(self, position, value):
        if position in self.matched:
            return
        self.seen[position] = value
        self.by_value[value].add(position)

    def mark_matched(self, pos1, pos2):
        self.matched.add(pos1)
        self.matched.add(pos2)

    def known_pair(self):
        for value, positions in self.by_value.items():
            candidates = [p for p in positions if p not in self.matched]
            if len(candidates) >= 2:
                return candidates[0], candidates[1], value
        return None

    def ucb_score(self, position):
        count = self.action_counts[position]
        if count == 0:
            return float('inf')
        mean_reward = self.action_values[position]
        bonus = self.c * ((self.total_actions ** 0.5) / (1 + count))
        return mean_reward + bonus

    def update_action(self, position, reward):
        self.total_actions += 1
        self.action_counts[position] += 1
        count = self.action_counts[position]
        old_value = self.action_values[position]
        self.action_values[position] = old_value + (reward - old_value) / count

    def choose_first(self, game):
        pair = self.known_pair()
        if pair is not None:
            return pair[0], 'explotar pareja conocida'

        hidden = game.hidden_positions()
        unseen = [p for p in hidden if p not in self.seen]
        candidate_pool = unseen if unseen else hidden
        best_position = max(candidate_pool, key=self.ucb_score)
        return best_position, 'seleccion UCB'

    def choose_second(self, game, first_position, first_value):
        same_value_positions = [
            p for p in self.by_value[first_value]
            if p != first_position and p not in self.matched
        ]
        if same_value_positions:
            return same_value_positions[0], 'explotar valor conocido'

        hidden = [p for p in game.hidden_positions() if p != first_position]
        unseen = [p for p in hidden if p not in self.seen]
        candidate_pool = unseen if unseen else hidden
        best_position = max(candidate_pool, key=self.ucb_score)
        return best_position, 'seleccion UCB'

    def memory_size(self):
        return len(self.seen)

    def summary(self):
        return {
            'seen_positions': len(self.seen),
            'known_values': len(self.by_value),
            'matched_positions': len(self.matched),
            'action_counts': dict(self.action_counts),
        }

In [11]:
class MemoryPairAgent:
    def __init__(self, epsilon=0.35, seed=None):
        self.epsilon = epsilon
        self.rng = random.Random(seed)
        self.reset()

    def reset(self):
        self.seen = {}
        self.by_value = defaultdict(set)
        self.matched = set()

    def observe(self, position, value):
        if position in self.matched:
            return
        self.seen[position] = value
        self.by_value[value].add(position)

    def mark_matched(self, pos1, pos2):
        self.matched.add(pos1)
        self.matched.add(pos2)

    def known_pair(self):
        for value, positions in self.by_value.items():
            candidates = [p for p in positions if p not in self.matched]
            if len(candidates) >= 2:
                return candidates[0], candidates[1], value
        return None

    def choose_first(self, game):
        pair = self.known_pair()
        if pair is not None:
            return pair[0], 'explotar pareja conocida'

        hidden = game.hidden_positions()
        unseen = [p for p in hidden if p not in self.seen]
        if unseen and self.rng.random() < self.epsilon:
            return self.rng.choice(unseen), 'explorar carta oculta'

        if unseen:
            return self.rng.choice(unseen), 'explorar carta nueva'

        return self.rng.choice(hidden), 'explorar por descarte'

    def choose_second(self, game, first_position, first_value):
        same_value_positions = [
            p for p in self.by_value[first_value]
            if p != first_position and p not in self.matched
        ]
        if same_value_positions:
            return same_value_positions[0], 'explotar valor conocido'

        hidden = [p for p in game.hidden_positions() if p != first_position]
        unseen = [p for p in hidden if p not in self.seen]
        if unseen and self.rng.random() < self.epsilon:
            return self.rng.choice(unseen), 'explorar segunda carta'

        if unseen:
            return self.rng.choice(unseen), 'explorar informacion nueva'

        return self.rng.choice(hidden), 'usar carta restante'

    def memory_size(self):
        return len(self.seen)

    def summary(self):
        return {
            'seen_positions': len(self.seen),
            'known_values': len(self.by_value),
            'matched_positions': len(self.matched),
        }

In [13]:
def play_one_game(agent, game, verbose=True):
    agent.reset()
    game.reset()
    turns = 0

    if verbose:
        print('Estado inicial:')
        game.draw()
        print('')

    while not game.done():
        first_position, first_reason = agent.choose_first(game)
        first_value = game.reveal(first_position)
        agent.observe(first_position, first_value)

        second_position, second_reason = agent.choose_second(game, first_position, first_value)
        second_value = game.reveal(second_position)
        agent.observe(second_position, second_value)

        is_pair = game.match(first_position, second_position)
        if is_pair:
            agent.mark_matched(first_position, second_position)

        turns += 1

        if verbose:
            print(f'Turno {turns}:')
            print(f'  primera carta  -> pos {first_position}, valor {first_value} ({first_reason})')
            print(f'  segunda carta   -> pos {second_position}, valor {second_value} ({second_reason})')
            print(f'  pareja encontrada: {is_pair}')
            print(f'  memoria actual   : {agent.summary()}')
            print('')

    if verbose:
        print('Partida terminada. Tablero final:')
        game.draw()

    return turns, agent.summary()

def run_experiment(n_games=3, n_pairs=8, seed=7, epsilon=0.35, verbose_first_game=True):
    rng = random.Random(seed)
    agent = MemoryPairAgent(epsilon=epsilon, seed=seed)
    results = []

    for index in range(n_games):
        game = MemoryPairsGame(n_pairs=n_pairs, seed=rng.randint(0, 10**9))
        verbose = verbose_first_game and index == 0
        turns, summary = play_one_game(agent, game, verbose=verbose)
        results.append({'game': index + 1, 'turns': turns, 'summary': summary})

    return agent, results

In [7]:
def play_one_game(agent, game, verbose=True):
    agent.reset()
    game.reset()
    turns = 0

    if verbose:
        print('Estado inicial:')
        game.draw()
        print('')

    while not game.done():
        first_position, first_reason = agent.choose_first(game)
        first_value = game.reveal(first_position)
        agent.observe(first_position, first_value)

        second_position, second_reason = agent.choose_second(game, first_position, first_value)
        second_value = game.reveal(second_position)
        agent.observe(second_position, second_value)

        is_pair = game.match(first_position, second_position)
        if is_pair:
            agent.mark_matched(first_position, second_position)

        turns += 1

        if verbose:
            print(f'Turno {turns}:')
            print(f'  primera carta  -> pos {first_position}, valor {first_value} ({first_reason})')
            print(f'  segunda carta   -> pos {second_position}, valor {second_value} ({second_reason})')
            print(f'  pareja encontrada: {is_pair}')
            print(f'  memoria actual   : {agent.summary()}')
            print('')

    if verbose:
        print('Partida terminada. Tablero final:')
        game.draw()

    return turns, agent.summary()

def run_experiment(n_games=3, n_pairs=8, seed=7, epsilon=0.35, verbose_first_game=True):
    rng = random.Random(seed)
    agent = MemoryPairAgent(epsilon=epsilon, seed=seed)
    results = []

    for index in range(n_games):
        game = MemoryPairsGame(n_pairs=n_pairs, seed=rng.randint(0, 10**9))
        verbose = verbose_first_game and index == 0
        turns, summary = play_one_game(agent, game, verbose=verbose)
        results.append({'game': index + 1, 'turns': turns, 'summary': summary})

    return agent, results

In [14]:
agent, results = run_experiment(n_games=3, n_pairs=8, seed=42, epsilon=0.4, verbose_first_game=True)

print('Resultados generales:')
for item in results:
    print(item)

print('Memoria final del agente:')
print(agent.summary())

NameError: name 'MemoryPairsGame' is not defined